In [1]:
import os
from dare2d.datamodule.generator._2d.regress_2dataset import Regress2Dataset
from dare2d.datamodule.generator._2d.seg_2dataset import Segmentation2Dataset
import albumentations as A
from dare2d.datamodule.augmentation import ElasticHistogram, Illumination
import cv2
import numpy as np

target_crop_size = 64
data_dir = "../data/2d"

def display(img, bipoint, ax):
    img = img * 255
    img = img.astype(np.uint8)
    ys = [bipoint[0][0], bipoint[1][0]]
    xs = [bipoint[0][1], bipoint[1][1]]
    
    # ax.plot(xs, ys, color="red", linewidth=3)
    ax.imshow(img, cmap="gray")
    # ax.imshow(mask, cmap='jet', alpha=0.5)


test_data_dir = os.path.join(data_dir, "set_1")

# test_generator = Segmentation2Dataset(test_data_dir, input_channels=[-1, 0, 1], crop_size=256, cell_size=4, type_mask='center', renorm="min-max")
test_generator = Regress2Dataset(test_data_dir, input_channels=[1], crop_size=64, renorm="min-max")

flip = A.Flip(p=0.5)
flip_with_noise = A.Compose([flip,
                                A.OneOf([
                                    A.GaussianBlur(blur_limit=23),
                                    A.GaussNoise(var_limit=30. / 255),
                                    A.MedianBlur(blur_limit=5)
                                ], p=0.5)
                                ])
brighten = A.RandomBrightnessContrast(
    p=0.5, brightness_limit=[.3, .5], contrast_limit=[.3, .5])  # , brightness_by_max=False)

random_crop = A.OneOf([
    A.RandomSizedCrop(min_max_height=(50, 101),
                        height=256, width=256, p=0.5),
], p=0.5)
distort = A.OneOf([
    A.GridDistortion(p=0.5),
    A.ElasticTransform(sigma=1, alpha_affine=20, p=0.5)
], p=0.5)
rotate = A.ShiftScaleRotate(
    p=0.5, shift_limit=0, scale_limit=0, rotate_limit=180, border_mode=cv2.BORDER_CONSTANT)
elastic = ElasticHistogram(num_control_points=5, max_repeat=1, p=0.5)
augmentations = A.Compose([brighten, flip_with_noise, rotate, distort, elastic, random_crop])


# train_generator.set_augmentations(augmentations)

import matplotlib.pyplot as plt
n_row_col = 6
n_img = n_row_col**2

def display_generator(generator):
    
    fig, axs = plt.subplots(n_row_col, n_row_col, figsize=(15,15))
    for i, (x, y) in enumerate(generator):
        col = i % n_row_col
        row = i // n_row_col
        bipoint = generator.bipoint_crops[i]
        
        generator.display(x, bipoint)
        
        # cv2.line(x[:, :, 0], np.array(bipoint[0]), np.array(bipoint[1]), 127, 4)
        display(x[:, :, 0], bipoint, axs[row, col])
        
        
        if i >= n_img - 1:
            break
    plt.show()
    
display_generator(test_generator)

2023-07-26 16:43:19.348346: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-26 16:43:19.451361: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-07-26 16:43:19.935395: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2023-07-26 16:43:19.935469: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] 

 LOADED 31 images with 1 timestamps
 -> image 0 shape: (768, 768, 1), dtype uint8, min-max: 0-255 
